# Part 1 — Crop Classification with MCTNet
### Paper: *A Lightweight CNN-Transformer Network for Pixel-Based Crop Mapping*
**Data:** Google Earth Engine export — 36 timesteps × 10 bands × 10,000 pixels  
**Study area:** Arkansas, USA, 2021

## 0. Install dependencies

In [6]:
import subprocess, sys
pkgs = ["numpy","pandas","matplotlib","seaborn","scikit-learn","torch","tqdm"]
subprocess.check_call([sys.executable,"-m","pip","install","-q"]+pkgs)
print("✅ All packages ready")

✅ All packages ready


## 1. Imports

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, accuracy_score, cohen_kappa_score)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi":120,"font.size":11})
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports OK | device: {device}")

✅ Imports OK | device: cpu


## 2. Configuration

In [8]:
# ── File path ─────────────────────────────────────────────────────────
ARKANSAS_CSV = r"C:\Users\HP\Desktop\SII\aarn\projet\MCTNet_Arkansas_2021_36t.csv"

# ── Paper settings ─────────────────────────────────────────────────────
N_TIMESTEPS     = 36
N_BANDS         = 10
TRAIN_PER_CLASS = 240
VAL_PER_CLASS   = 60

# ── CDL crop codes for Arkansas (paper Table 2) ────────────────────────
CDL_ARKANSAS = {
    5: "Soybeans",
    3: "Rice",
    1: "Corn",
    2: "Cotton",
}

# ── Model hyperparameters (paper Table 3 — Arkansas) ───────────────────
HIDDEN_DIM  = 64
N_HEADS     = 5
N_STAGES    = 3
KERNEL_SIZE = 3
DROPOUT     = 0.1
BATCH_SIZE  = 32
EPOCHS      = 200
LR          = 0.001

print("✅ Config ready")
print(f"   Arkansas CSV: {ARKANSAS_CSV}")

✅ Config ready
   Arkansas CSV: C:\Users\HP\Desktop\SII\aarn\projet\MCTNet_Arkansas_2021_36t.csv


## 3. Load dataset

In [9]:
def load_csv(path):
    df = pd.read_csv(path)
    print(f"Loaded: {path}")
    print(f"  Shape     : {df.shape}")
    print(f"  Columns   : {list(df.columns[:6])} ...")

    drop = [c for c in df.columns
            if c.startswith('system') or c.startswith('.geo')]
    label_col = 'cropland'
    feat_cols = [c for c in df.columns
                 if c not in drop and c != label_col]

    X = df[feat_cols].fillna(0).values.astype(np.float32)
    y = df[label_col].fillna(0).values.astype(np.int32)

    print(f"  Features  : {X.shape}  ({X.shape[1]} columns = "
          f"{N_TIMESTEPS} timesteps x {N_BANDS} bands)")
    print(f"  Labels    : {y.shape}")
    print(f"  Crop codes: {sorted(np.unique(y))}")
    return X, y, df, feat_cols

X_ar, y_ar, df_ar, feat_cols = load_csv(ARKANSAS_CSV)

EmptyDataError: No columns to parse from file

## 4. Reshape features to (N, T, C)

In [ ]:
def reshape_to_timeseries(X, n_timesteps=36, n_bands=10):
    N = X.shape[0]
    X_3d = X.reshape(N, n_timesteps, n_bands)
    print(f"Reshaped: {X.shape} → {X_3d.shape}  (samples, timesteps, bands)")
    return X_3d

X_3d = reshape_to_timeseries(X_ar, N_TIMESTEPS, N_BANDS)
print(f"Band order per timestep: {feat_cols[:10]}")

## 5. Data Exploration
### 5.1 Class distribution

In [ ]:
def plot_class_distribution(y, cdl_classes, title):
    counts = Counter(y.tolist())
    named = {}
    for code, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        name = cdl_classes.get(int(code), f"Other ({int(code)})")
        named[name] = named.get(name, 0) + cnt

    fig, ax = plt.subplots(figsize=(11, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(named)))
    bars = ax.bar(list(named.keys()), list(named.values()),
                  color=colors, edgecolor='white', linewidth=0.5)
    ax.bar_label(bars, fmt='%d', fontsize=9, padding=3)
    ax.set(xlabel='Crop Type', ylabel='Number of Pixels',
           title=f'Class Distribution — {title}')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('viz_class_dist_ar.png', dpi=150)
    plt.show()
    print(f"Total pixels: {len(y)}")

plot_class_distribution(y_ar, CDL_ARKANSAS, "Arkansas 2021")

### 5.2 NDVI time-series per crop (paper Figure 2)

In [ ]:
def compute_ndvi(X_3d):
    # Band order from GEE (alphabetical): B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A
    # Red=B4 → index 4,  NIR=B8 → index 8
    red = X_3d[:, :, 4].astype(float)
    nir = X_3d[:, :, 8].astype(float)
    return (nir - red) / (nir + red + 1e-8)

def plot_ndvi_timeseries(X_3d, y, cdl_classes, title):
    ndvi = compute_ndvi(X_3d)
    doy  = np.arange(5, 5 + N_TIMESTEPS * 10, 10)

    fig, ax = plt.subplots(figsize=(13, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(cdl_classes)))

    for (code, name), color in zip(cdl_classes.items(), colors):
        mask = (y == code)
        if mask.sum() < 5:
            continue
        mean_ndvi = ndvi[mask].mean(axis=0)
        std_ndvi  = ndvi[mask].std(axis=0)
        ax.plot(doy, mean_ndvi, 'o-', label=name,
                color=color, lw=2, ms=4)
        ax.fill_between(doy,
                        mean_ndvi - std_ndvi,
                        mean_ndvi + std_ndvi,
                        alpha=0.1, color=color)

    ax.set(xlabel='Day of Year', ylabel='Mean NDVI',
           title=f'NDVI Time-Series Profiles — {title}',
           ylim=(-0.1, 1.0))
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.legend(bbox_to_anchor=(1.01, 1), fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('viz_ndvi_ar.png', dpi=150)
    plt.show()

plot_ndvi_timeseries(X_3d, y_ar, CDL_ARKANSAS, "Arkansas 2021")

### 5.3 Temporal NIR and Red band patterns per crop

In [ ]:
def plot_temporal_band(X_3d, y, cdl_classes, band_idx, band_name, title):
    doy = np.arange(5, 5 + N_TIMESTEPS * 10, 10)
    fig, ax = plt.subplots(figsize=(13, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(cdl_classes)))

    for (code, name), color in zip(cdl_classes.items(), colors):
        mask = (y == code)
        if mask.sum() < 5:
            continue
        vals = X_3d[mask, :, band_idx].astype(float)
        vals[vals == 0] = np.nan
        mean_vals = np.nanmean(vals, axis=0)
        ax.plot(doy, mean_vals, 'o-', label=name,
                color=color, lw=2, ms=4)

    ax.set(xlabel='Day of Year',
           ylabel=f'Mean {band_name} Reflectance',
           title=f'{band_name} Temporal Profile by Crop — {title}')
    ax.legend(bbox_to_anchor=(1.01, 1), fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'viz_temporal_{band_name.split()[0]}_ar.png', dpi=150)
    plt.show()

plot_temporal_band(X_3d, y_ar, CDL_ARKANSAS, 8, 'NIR (B8)', 'Arkansas')
plot_temporal_band(X_3d, y_ar, CDL_ARKANSAS, 4, 'Red (B4)', 'Arkansas')

### 5.4 Missing values analysis

In [ ]:
def plot_missing_values(X_3d, title):
    missing_per_sample   = np.zeros(X_3d.shape[0])
    missing_per_timestep = np.zeros(N_TIMESTEPS)
    doy = np.arange(5, 5 + N_TIMESTEPS * 10, 10)

    for t in range(N_TIMESTEPS):
        all_zero = (X_3d[:, t, :] == 0).all(axis=1)
        missing_per_sample   += all_zero.astype(float)
        missing_per_timestep[t] = all_zero.mean() * 100

    missing_rate = missing_per_sample / N_TIMESTEPS * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.hist(missing_rate, bins=20, color='#378ADD',
             edgecolor='white', linewidth=0.5)
    ax1.axvline(missing_rate.mean(), color='red', ls='--', lw=1.5,
                label=f'Mean: {missing_rate.mean():.1f}%')
    ax1.set(xlabel='Missing timesteps (%)', ylabel='Number of pixels',
            title=f'Missing Data Distribution — {title}')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.bar(doy, missing_per_timestep, width=8,
            color='#E63946', edgecolor='white', linewidth=0.3)
    ax2.set(xlabel='Day of Year', ylabel='Missing rate (%)',
            title=f'Missing Rate per Timestep — {title}')
    ax2.grid(alpha=0.3)

    plt.suptitle(f'Missing Values Analysis — {title}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('viz_missing_ar.png', dpi=150)
    plt.show()

    print(f"Average missing rate : {missing_rate.mean():.1f}%")
    print(f"Pixels >30% missing  : {(missing_rate>30).sum()}")
    print(f"Pixels 0% missing    : {(missing_rate==0).sum()}")

plot_missing_values(X_3d, "Arkansas 2021")

## 6. Preprocessing

In [ ]:
def build_missing_mask(X_3d):
    mask = np.ones((X_3d.shape[0], N_TIMESTEPS), dtype=np.float32)
    for t in range(N_TIMESTEPS):
        all_zero = (X_3d[:, t, :] == 0).all(axis=1)
        mask[all_zero, t] = 0
    return mask

def normalize(X_3d):
    flat = X_3d.reshape(-1, N_BANDS)
    mean = flat.mean(axis=0)
    std  = flat.std(axis=0) + 1e-8
    return (X_3d - mean) / std, mean, std

def split_dataset(X_3d, mask, y, cdl_classes,
                  train_per=240, val_per=60):
    valid_codes = list(cdl_classes.keys())
    y_mapped = np.where(np.isin(y, valid_codes), y, 0)
    all_codes = sorted(np.unique(y_mapped))
    le = LabelEncoder()
    le.fit(all_codes)
    y_enc = le.transform(y_mapped)

    np.random.seed(42)
    tr_idx, va_idx, te_idx = [], [], []
    for cls_code in all_codes:
        cls_label = le.transform([cls_code])[0]
        idx = np.where(y_enc == cls_label)[0]
        np.random.shuffle(idx)
        n_tv = train_per + val_per
        if len(idx) >= n_tv:
            tr_idx.extend(idx[:train_per])
            va_idx.extend(idx[train_per:n_tv])
            te_idx.extend(idx[n_tv:])
        else:
            t1 = int(len(idx)*0.7)
            t2 = int(len(idx)*0.85)
            tr_idx.extend(idx[:t1])
            va_idx.extend(idx[t1:t2])
            te_idx.extend(idx[t2:])

    splits = {}
    for name, idx in [('train',tr_idx),('val',va_idx),('test',te_idx)]:
        splits[name] = {'X':X_3d[idx],'mask':mask[idx],'y':y_enc[idx]}
        print(f"  {name:6s}: {len(idx):5d} samples")
    return splits, le, all_codes

mask_ar = build_missing_mask(X_3d)
X_norm, feat_mean, feat_std = normalize(X_3d)
print("\nSplit summary:")
splits_ar, le_ar, codes_ar = split_dataset(
    X_norm, mask_ar, y_ar, CDL_ARKANSAS)
class_names = [CDL_ARKANSAS.get(c, 'Others') for c in codes_ar]
print(f"\nClasses: {class_names}")

## 7. MCTNet Model

In [ ]:
class ALPE(nn.Module):
    def __init__(self, n_timesteps, d_model):
        super().__init__()
        pe = torch.zeros(n_timesteps, d_model)
        pos = torch.arange(n_timesteps).unsqueeze(1).float()
        div = torch.exp(torch.arange(0,d_model,2).float() *
                        -(np.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe)
        self.conv    = nn.Conv1d(d_model, d_model, 3, padding=1)
        self.eca     = nn.Conv1d(1, 1, 3, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, mask):
        B,T,D = x.shape
        pe = self.pe.unsqueeze(0).expand(B,-1,-1)
        masked_pe = pe * mask.unsqueeze(-1)
        out = self.conv(masked_pe.permute(0,2,1)).permute(0,2,1)
        gap = out.mean(dim=2,keepdim=True).permute(0,2,1)
        attn = self.sigmoid(self.eca(gap)).permute(0,2,1)
        return out * attn

class CNNBlock(nn.Module):
    def __init__(self, d, k=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(d,d,k,padding=k//2), nn.BatchNorm1d(d),
            nn.Conv1d(d,d,k,padding=k//2), nn.BatchNorm1d(d),
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.net(x.permute(0,2,1)).permute(0,2,1) + x)

class TransformerBlock(nn.Module):
    def __init__(self, d, heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, heads,
                                            dropout=dropout,
                                            batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d,d*4), nn.GELU(), nn.Linear(d*4,d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        a,_ = self.attn(x,x,x)
        x   = self.norm1(x + self.drop(a))
        return self.norm2(x + self.drop(self.ff(x)))

class CTFusion(nn.Module):
    def __init__(self, d, heads, k=3, dropout=0.1):
        super().__init__()
        self.cnn   = CNNBlock(d, k)
        self.trans = TransformerBlock(d, heads, dropout)
        self.pool  = nn.MaxPool1d(2, stride=2)
    def forward(self, x, pos=None):
        c = self.cnn(x)
        t = self.trans(x + pos if pos is not None else x)
        fused = torch.cat([c, t], dim=-1)
        return self.pool(fused.permute(0,2,1)).permute(0,2,1)

class MCTNet(nn.Module):
    def __init__(self, n_bands, n_timesteps, n_classes,
                 d=64, heads=5, stages=3, k=3, dropout=0.1):
        super().__init__()
        self.proj   = nn.Linear(n_bands, d)
        self.alpe   = ALPE(n_timesteps, d)
        self.stages = nn.ModuleList()
        for _ in range(stages):
            self.stages.append(CTFusion(d, heads, k, dropout))
            d = d * 2
        self.head = nn.Linear(d, n_classes)
    def forward(self, x, mask):
        x   = self.proj(x)
        pos = self.alpe(x, mask)
        for i, stage in enumerate(self.stages):
            x = stage(x, pos if i==0 else None)
            if i == 0:
                pos = pos[:, ::2, :]
        return self.head(x.max(dim=1)[0])

n_classes = len(codes_ar)
model = MCTNet(N_BANDS, N_TIMESTEPS, n_classes,
               d=HIDDEN_DIM, heads=N_HEADS,
               stages=N_STAGES, k=KERNEL_SIZE,
               dropout=DROPOUT).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ MCTNet ready")
print(f"   Parameters : {n_params:,}  (paper: 55,059)")
print(f"   Classes    : {n_classes}")

## 8. Training

In [ ]:
def make_loader(split_data, shuffle=False):
    X = torch.tensor(split_data['X'],    dtype=torch.float32)
    m = torch.tensor(split_data['mask'], dtype=torch.float32)
    y = torch.tensor(split_data['y'],    dtype=torch.long)
    return DataLoader(TensorDataset(X,m,y),
                      batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(splits_ar['train'], shuffle=True)
val_loader   = make_loader(splits_ar['val'])
test_loader  = make_loader(splits_ar['test'])

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

history = {"train_loss":[],"val_loss":[],"val_f1":[]}
best_f1  = 0.0

print(f"Training MCTNet — Arkansas — {EPOCHS} epochs on {device}")
print(f"Train: {len(splits_ar['train']['y'])} | "
      f"Val: {len(splits_ar['val']['y'])} | "
      f"Test: {len(splits_ar['test']['y'])}")

for epoch in range(1, EPOCHS+1):
    model.train()
    t_loss = []
    for xb,mb,yb in train_loader:
        xb,mb,yb = xb.to(device),mb.to(device),yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb,mb), yb)
        loss.backward(); optimizer.step()
        t_loss.append(loss.item())

    model.eval()
    v_loss, preds, trues = [], [], []
    with torch.no_grad():
        for xb,mb,yb in val_loader:
            xb,mb,yb = xb.to(device),mb.to(device),yb.to(device)
            out = model(xb,mb)
            v_loss.append(criterion(out,yb).item())
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.cpu().numpy())

    scheduler.step()
    tl = np.mean(t_loss); vl = np.mean(v_loss)
    vf = f1_score(trues, preds, average='macro', zero_division=0)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_f1'].append(vf)

    if vf > best_f1:
        best_f1 = vf
        torch.save(model.state_dict(), 'mctnet_arkansas_best.pt')

    if epoch % 25 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{EPOCHS} | "
              f"train={tl:.4f}  val={vl:.4f}  F1={vf:.4f}")

model.load_state_dict(
    torch.load('mctnet_arkansas_best.pt', map_location=device))
print(f"\n✅ Training done | Best val F1: {best_f1:.4f}")

### 8.1 Learning curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['train_loss'], label='Train', color='#E63946', lw=2)
ax1.plot(history['val_loss'],   label='Val',   color='#457B9D', lw=2)
ax1.set(title='Loss Curves', xlabel='Epoch', ylabel='Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['val_f1'], color='#2A9D8F', lw=2)
ax2.set(title='Validation Macro F1', xlabel='Epoch', ylabel='F1 Score')
ax2.grid(alpha=0.3)

plt.suptitle('Training History — MCTNet Arkansas', fontweight='bold')
plt.tight_layout()
plt.savefig('viz_training_curves_ar.png', dpi=150)
plt.show()

## 9. Evaluation

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb,mb,yb in test_loader:
        xb,mb = xb.to(device),mb.to(device)
        all_preds.extend(model(xb,mb).argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

oa    = accuracy_score(all_true, all_preds)
kappa = cohen_kappa_score(all_true, all_preds)
f1    = f1_score(all_true, all_preds, average='macro', zero_division=0)

print("="*55)
print("  Arkansas — Test Results")
print("="*55)
print(f"  Overall Accuracy (OA) : {oa:.4f}  (paper: 0.968)")
print(f"  Kappa coefficient     : {kappa:.4f}  (paper: 0.951)")
print(f"  Macro F1-Score        : {f1:.4f}  (paper: 0.933)")
print("="*55)
print()
print(classification_report(all_true, all_preds,
      target_names=class_names, zero_division=0))

### 9.1 Confusion matrix

In [ ]:
cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            ax=ax, linewidths=0.5)
ax.set(xlabel='Predicted', ylabel='True',
       title='Confusion Matrix — Arkansas Test Set')
plt.tight_layout()
plt.savefig('viz_confusion_matrix_ar.png', dpi=150)
plt.show()

### 9.2 Per-class F1 score

In [ ]:
per_class_f1 = f1_score(all_true, all_preds,
                         average=None, zero_division=0)
fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(class_names)))
bars = ax.bar(class_names[:len(per_class_f1)],
              per_class_f1, color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%.3f', fontsize=9, padding=3)
ax.axhline(f1, color='red', ls='--', lw=1.5,
           label=f'Macro F1 = {f1:.3f}')
ax.set(ylim=(0, 1.15), xlabel='Crop Class',
       ylabel='F1 Score',
       title='Per-Class F1 Score — Arkansas')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('viz_per_class_f1_ar.png', dpi=150)
plt.show()

## 10. Results Summary

In [ ]:
print("\n" + "="*55)
print("  FINAL RESULTS vs PAPER TABLE 5 — ARKANSAS")
print("="*55)
print(f"  {'Metric':<20} {'Your result':<15} {'Paper'}")
print("-"*55)
print(f"  {'Overall Accuracy':<20} {oa:<15.4f} 0.968")
print(f"  {'Kappa':<20} {kappa:<15.4f} 0.951")
print(f"  {'Macro F1':<20} {f1:<15.4f} 0.933")
print(f"  {'Parameters':<20} {sum(p.numel() for p in model.parameters()):<15,} 55,059")
print("="*55)